In [0]:
%pip install faker


In [0]:
dbutils.library.restartPython()

In [0]:
import random, hashlib
from datetime import date, timedelta, datetime, timezone
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from faker import Faker
 
fake = Faker("en_IN")
random.seed(42)
np.random.seed(42)
 
DB = "hospital_blackhole"
spark.sql(f"USE {DB}")


In [0]:
STATES = {
    "Jharkhand"   :{"code":"JH","districts":24,"pop_m":3.8, "tribal_pct":0.26},
    "Bihar"       :{"code":"BR","districts":38,"pop_m":12.4,"tribal_pct":0.01},
    "Uttar Pradesh":{"code":"UP","districts":75,"pop_m":23.4,"tribal_pct":0.00},
    "Odisha"      :{"code":"OD","districts":30,"pop_m":4.2, "tribal_pct":0.23},
    "Madhya Pradesh":{"code":"MP","districts":52,"pop_m":8.5,"tribal_pct":0.21},
    "Rajasthan"   :{"code":"RJ","districts":33,"pop_m":8.1, "tribal_pct":0.14},
    "Chhattisgarh":{"code":"CG","districts":27,"pop_m":2.9, "tribal_pct":0.32},
    "Assam"       :{"code":"AS","districts":33,"pop_m":3.5, "tribal_pct":0.13},
    "West Bengal" :{"code":"WB","districts":23,"pop_m":9.8, "tribal_pct":0.05},
    "Maharashtra" :{"code":"MH","districts":36,"pop_m":12.8,"tribal_pct":0.09},
}
 
FACILITY_TYPES = {
    "Sub Centre"       :{"abbr":"SC","level":1,"beds":0,"specialists":0},
    "Primary HC"       :{"abbr":"PHC","level":2,"beds":6,"specialists":0},
    "Community HC"     :{"abbr":"CHC","level":3,"beds":30,"specialists":4},
    "District Hospital":{"abbr":"DH","level":4,"beds":200,"specialists":12},
    "Medical College"  :{"abbr":"MC","level":5,"beds":500,"specialists":40},
}
 
DISEASE_CATS = [
    "Cancer","Tuberculosis","Cardiovascular","Diabetes",
    "Anaemia","Maternal","Mental Health","Injury & Trauma"
]
 
SPECIALTY_MAP = {
    "Oncology":["Cancer"],"Pulmonology":["Tuberculosis"],
    "Cardiology":["Cardiovascular"],"Endocrinology":["Diabetes"],
    "Haematology":["Anaemia"],"Gynaecology":["Maternal"],
    "Psychiatry":["Mental Health"],"Orthopaedics":["Injury & Trauma"],
}


In [0]:
districts = []
dist_id = 1
for state, meta in STATES.items():
    n = min(5, meta["districts"])  # 5 districts per state = 50 total
    for d in range(n):
        dist_code = f"{meta['code']}{d+1:02d}"
        pop = int((meta["pop_m"] * 1e6 / meta["districts"]) * random.uniform(0.5, 1.8))
        # Vulnerability: tribal + rural + poor districts are worse
        vuln = round(
            (meta["tribal_pct"] * 0.4) +
            (random.uniform(0, 0.6)) +
            (random.uniform(0, 0.3) if meta["code"] in ["JH","BR","OD","CG"] else 0),
            3
        )
        districts.append({
            "district_id"     : dist_id,
            "district_code"   : dist_code,
            "district_name"   : f"{state[:4]}_District_{d+1}",
            "state"           : state,
            "state_code"      : meta["code"],
            "population"      : pop,
            "vulnerability_idx": min(vuln, 1.0),
            "tribal_pct"      : meta["tribal_pct"],
            "lat"             : round(random.uniform(18, 28), 4),
            "lon"             : round(random.uniform(74, 90), 4),
        })
        dist_id += 1
 
# Build facility master
facilities = []
fac_id = 1
for dist in districts:
    # Facilities per district based on population
    pop = dist["population"]
    n_sc   = max(2, int(pop / 5000))
    n_phc  = max(1, int(pop / 30000))
    n_chc  = max(1, int(pop / 120000))
    n_dh   = 1
    n_mc   = 1 if dist["vulnerability_idx"] < 0.3 else 0
 
    for ftype, counts in [
        ("Sub Centre", n_sc), ("Primary HC", n_phc),
        ("Community HC", n_chc), ("District Hospital", n_dh),
        ("Medical College", n_mc)
    ]:
        for i in range(counts):
            meta = FACILITY_TYPES[ftype]
            # Vacancy rate is higher in vulnerable districts
            vacancy = round(random.uniform(0.1, 0.6) * (1 + dist["vulnerability_idx"]), 2)
            actual_spec = max(0, int(meta["specialists"] * (1 - vacancy)))
            facilities.append({
                "facility_id"        : f"FAC{fac_id:06d}",
                "facility_type"      : ftype,
                "facility_level"     : meta["level"],
                "district_id"        : dist["district_id"],
                "district_name"      : dist["district_name"],
                "state"              : dist["state"],
                "beds_sanctioned"    : meta["beds"],
                "beds_functional"    : max(0, int(meta["beds"] * random.uniform(0.6, 1.0))),
                "specialists_sanctioned": meta["specialists"],
                "specialists_functional": actual_spec,
                "vacancy_rate"       : vacancy,
                "distance_to_dh_km" : round(random.uniform(5, 120) * (1 + dist["vulnerability_idx"] * 0.5), 1),
                "travel_hours_to_dh" : round(random.uniform(0.5, 8) * (1 + dist["vulnerability_idx"] * 0.3), 2),
                "functional_status"  : "Functional" if random.random() > 0.1 else "Partially Functional",
                "ingested_at"        : str(datetime.now(timezone.utc)),
            })
            fac_id += 1
 
print(f"Facilities generated: {len(facilities):,} across {len(districts)} districts")
 
df_fac = spark.createDataFrame(pd.DataFrame(facilities))
df_dist = spark.createDataFrame(pd.DataFrame(districts))
 
df_fac.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.bronze_facility_master")
df_dist.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.bronze_district_master")
 
print("✅ Facility master and district master written")
 

In [0]:
# ============================================================
# SECTION B — HMIS REFERRAL DATA
# Based on NHM HMIS:
#   - ~85M OPD cases in PHCs monthly (all India)
#   - ~30% require referral upward
#   - Referral acceptance at CHC: 53–70% based on state
# We generate 3 years of monthly data per district
# ============================================================
 
# COMMAND ----------
# Dropout probabilities seeded from real NHM/HMIS research
# Source: Lancet India 2023, PHFI reports, NHM HMIS handbooks
DROPOUT_RATES = {
    # stage: (base_rate, vulnerability_multiplier)
    # stage 1 = PHC → referral issued
    "stage1_no_referral"   : (0.18, 0.12),   # 18–30% no referral issued
    # stage 2 = referral issued → CHC attended
    "stage2_no_attend"     : (0.25, 0.18),   # 25–43% don't attend CHC
    # stage 3 = CHC → specialist referral
    "stage3_no_specialist" : (0.15, 0.10),   # 15–25% stop at CHC
    # stage 4 = specialist ref → tertiary hospital
    "stage4_no_tertiary"   : (0.10, 0.08),   # 10–18% don't reach tertiary
    # stage 5 = tertiary → treatment started
    "stage5_no_treatment"  : (0.06, 0.04),   # 6–10% dropout at admission
    # stage 6 = treatment started → completed
    "stage6_incomplete"    : (0.05, 0.03),   # 5–8% treatment dropout
}
 
DISEASE_SEVERITY = {
    "Cancer":1.4, "Tuberculosis":1.2, "Cardiovascular":1.1,
    "Diabetes":0.9, "Anaemia":0.7, "Maternal":0.8,
    "Mental Health":1.5, "Injury & Trauma":0.9
}
 
# Monthly incidence rates per 100K population (from ICMR/NFHS-5)
DISEASE_INCIDENCE_PER_100K = {
    "Cancer":120, "Tuberculosis":210, "Cardiovascular":280,
    "Diabetes":890, "Anaemia":1200, "Maternal":340,
    "Mental Health":180, "Injury & Trauma":150
}
 
hmis_records = []
rec_id = 1
start_month = date(2022, 1, 1)
 
for dist in districts:
    vuln = dist["vulnerability_idx"]
    pop = dist["population"]
 
    for month_offset in range(36):  # 3 years monthly
        report_month = start_month + timedelta(days=30 * month_offset)
 
        for disease in DISEASE_CATS:
            incidence_rate = DISEASE_INCIDENCE_PER_100K[disease] / 100000
            # Seasonal multiplier for certain diseases
            season_mult = 1.0
            if disease == "Tuberculosis" and report_month.month in [1, 2, 3]:
                season_mult = 1.3
            if disease == "Cardiovascular" and report_month.month in [5, 6]:
                season_mult = 1.2
            if disease == "Maternal":
                season_mult = random.uniform(0.8, 1.3)
 
            monthly_cases = int(pop * incidence_rate * season_mult / 12 * random.uniform(0.85, 1.15))
            if monthly_cases == 0:
                continue
 
            # Apply dropout cascade
            def dropout(n, base, vuln_mult, vuln):
                rate = min(base + vuln * vuln_mult, 0.75)
                dropped = int(n * rate * random.uniform(0.88, 1.12))
                return max(0, n - dropped), dropped
 
            at_phc = monthly_cases
            at_referred, d1 = dropout(at_phc, *DROPOUT_RATES["stage1_no_referral"], vuln)
            at_chc, d2 = dropout(at_referred, *DROPOUT_RATES["stage2_no_attend"], vuln)
            at_spec_ref, d3 = dropout(at_chc, *DROPOUT_RATES["stage3_no_specialist"], vuln)
            at_tertiary, d4 = dropout(at_spec_ref, *DROPOUT_RATES["stage4_no_tertiary"], vuln)
            at_treatment, d5 = dropout(at_tertiary, *DROPOUT_RATES["stage5_no_treatment"], vuln)
            completed, d6 = dropout(at_treatment, *DROPOUT_RATES["stage6_incomplete"], vuln)
 
            hmis_records.append({
                "record_id"           : f"HMIS{rec_id:08d}",
                "district_id"         : dist["district_id"],
                "district_name"       : dist["district_name"],
                "state"               : dist["state"],
                "state_code"          : dist["state_code"],
                "disease_category"    : disease,
                "report_month"        : str(report_month),
                "report_year"         : report_month.year,
                "cases_diagnosed_phc" : at_phc,
                "cases_referred"      : at_referred,
                "cases_chc_attended"  : at_chc,
                "cases_specialist_ref": at_spec_ref,
                "cases_tertiary"      : at_tertiary,
                "cases_treatment_started": at_treatment,
                "cases_completed"     : completed,
                "dropout_stage1"      : d1,
                "dropout_stage2"      : d2,
                "dropout_stage3"      : d3,
                "dropout_stage4"      : d4,
                "dropout_stage5"      : d5,
                "dropout_stage6"      : d6,
                "total_dropout"       : d1+d2+d3+d4+d5+d6,
                "vulnerability_idx"   : vuln,
                "population"          : pop,
                "ingested_at"         : str(datetime.now(timezone.utc)),
            })
            rec_id += 1
 
print(f"HMIS records generated: {len(hmis_records):,}")
 
df_hmis = spark.createDataFrame(pd.DataFrame(hmis_records))
df_hmis.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .partitionBy("state_code","report_year") \
    .saveAsTable(f"{DB}.bronze_hmis_referrals")
print("✅ Bronze HMIS referrals written")


In [0]:
# SECTION C — PMJAY CLAIMS DATA
# Based on NHA PMJAY Annual Report 2023:
#   - 2.43 crore hospitalisations (all time)
#   - 15,000+ empanelled hospitals
#   - Avg claim: ₹14,800
# We generate district-level monthly claim summaries
# ============================================================
 
pmjay_records = []
for dist in districts:
    vuln = dist["vulnerability_idx"]
    pop = dist["population"]
    # Utilisation inversely correlated with vulnerability (sad but true)
    utilisation_rate = max(0.02, 0.15 - (vuln * 0.08))
 
    for month_offset in range(36):
        report_month = start_month + timedelta(days=30 * month_offset)
        for disease in DISEASE_CATS:
            incidence = DISEASE_INCIDENCE_PER_100K[disease] / 100000 / 12
            expected = int(pop * incidence)
            claimed = int(expected * utilisation_rate * random.uniform(0.7, 1.3))
            avg_claim_inr = {
                "Cancer": 45000, "Tuberculosis": 8500, "Cardiovascular": 52000,
                "Diabetes": 12000, "Anaemia": 6500, "Maternal": 9800,
                "Mental Health": 15000, "Injury & Trauma": 28000
            }[disease]
 
            pmjay_records.append({
                "district_id"         : dist["district_id"],
                "state"               : dist["state"],
                "disease_category"    : disease,
                "report_month"        : str(report_month),
                "cases_expected"      : expected,
                "claims_submitted"    : claimed,
                "claims_approved"     : int(claimed * random.uniform(0.72, 0.92)),
                "avg_claim_inr"       : avg_claim_inr + int(random.gauss(0, avg_claim_inr * 0.1)),
                "claim_gap_pct"       : round((expected - claimed) / max(expected, 1) * 100, 2),
                "utilisation_rate"    : round(utilisation_rate, 4),
                "ingested_at"         : str(datetime.now(timezone.utc)),
            })
 
df_pmjay = spark.createDataFrame(pd.DataFrame(pmjay_records))
df_pmjay.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.bronze_pmjay_claims")
print(f"✅ PMJAY claims written: {len(pmjay_records):,} records")
 


In [0]:
# ============================================================
# SECTION D — NFHS-5 DISEASE BASELINE
# District-level prevalence from NFHS-5 open data
# ============================================================
 
# COMMAND ----------
nfhs_records = []
NFHS_BASELINE = {
    # (mean, std) for each state category
    "anaemia_women_pct"    : {"high_vuln": (68, 8),  "low_vuln": (48, 10)},
    "stunting_under5_pct"  : {"high_vuln": (42, 7),  "low_vuln": (28, 8)},
    "wasting_under5_pct"   : {"high_vuln": (22, 5),  "low_vuln": (14, 4)},
    "hypertension_m_pct"   : {"high_vuln": (26, 6),  "low_vuln": (24, 5)},
    "diabetes_pct"         : {"high_vuln": (11, 3),  "low_vuln": (14, 4)},
    "tb_notification_per_lakh": {"high_vuln": (280, 60), "low_vuln": (170, 40)},
}
 
for dist in districts:
    cat = "high_vuln" if dist["vulnerability_idx"] > 0.4 else "low_vuln"
    row = {"district_id": dist["district_id"], "district_name": dist["district_name"],
           "state": dist["state"], "vulnerability_idx": dist["vulnerability_idx"]}
    for metric, stats in NFHS_BASELINE.items():
        mean, std = stats[cat]
        row[metric] = round(max(1, random.gauss(mean, std)), 2)
    row["ingested_at"] = str(datetime.now(timezone.utc))
    nfhs_records.append(row)
 
df_nfhs = spark.createDataFrame(pd.DataFrame(nfhs_records))
df_nfhs.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.bronze_nfhs_baseline")
print(f"✅ NFHS-5 baseline written: {len(nfhs_records)} districts")


In [0]:
# ============================================================
# SECTION E — PATIENT-LEVEL COHORT DATA (for survival analysis)
# Synthetic individual patient events — state machine events
# Each row = one state transition for one patient cohort
# ============================================================
 
# COMMAND ----------
random.seed(99)
patient_events = []
event_id = 1
 
JOURNEY_STATES = [
    "DIAGNOSED","REFERRED","CHC_ATTENDED","SPECIALIST_SEEN",
    "TERTIARY_ADMITTED","TREATMENT_STARTED","TREATMENT_COMPLETED","DROPPED_OUT"
]
 
for dist in districts[:30]:  # 30 districts for cohort data (perf constraint)
    vuln = dist["vulnerability_idx"]
    for disease in DISEASE_CATS[:5]:  # top 5 diseases
        severity = DISEASE_SEVERITY[disease]
 
        # Generate 50-200 synthetic patient cohorts per district-disease
        n_cohorts = random.randint(50, 150)
        cohort_start = date(2022, 1, 1)
 
        for c in range(n_cohorts):
            patient_id = f"PT{event_id:09d}"
            diagnosis_date = cohort_start + timedelta(
                days=random.randint(0, 1000)
            )
            current_state = "DIAGNOSED"
            current_date = diagnosis_date
            dropped = False
 
            patient_events.append({
                "patient_id"      : patient_id,
                "district_id"     : dist["district_id"],
                "state"           : dist["state"],
                "disease"         : disease,
                "event_type"      : "DIAGNOSED",
                "event_date"      : str(current_date),
                "days_from_diag"  : 0,
                "is_terminal"     : False,
                "is_dropout"      : False,
                "vulnerability"   : vuln,
            })
 
            stage_probs = {
                "DIAGNOSED→REFERRED":    max(0.3, 0.82 - vuln * 0.35),
                "REFERRED→CHC_ATTENDED": max(0.3, 0.74 - vuln * 0.28),
                "CHC_ATTENDED→SPECIALIST_SEEN": max(0.4, 0.82 - vuln * 0.18),
                "SPECIALIST_SEEN→TERTIARY_ADMITTED": max(0.5, 0.88 - vuln * 0.12),
                "TERTIARY_ADMITTED→TREATMENT_STARTED": max(0.6, 0.92 - vuln * 0.08),
                "TREATMENT_STARTED→TREATMENT_COMPLETED": max(0.65, 0.94 - vuln * 0.06),
            }
            stage_days = {
                "DIAGNOSED→REFERRED":    (3, 45),
                "REFERRED→CHC_ATTENDED": (1, 30),
                "CHC_ATTENDED→SPECIALIST_SEEN": (7, 60),
                "SPECIALIST_SEEN→TERTIARY_ADMITTED": (14, 90),
                "TERTIARY_ADMITTED→TREATMENT_STARTED": (3, 21),
                "TREATMENT_STARTED→TREATMENT_COMPLETED": (14, 365),
            }
 
            transitions = [
                ("DIAGNOSED","REFERRED"),("REFERRED","CHC_ATTENDED"),
                ("CHC_ATTENDED","SPECIALIST_SEEN"),("SPECIALIST_SEEN","TERTIARY_ADMITTED"),
                ("TERTIARY_ADMITTED","TREATMENT_STARTED"),("TREATMENT_STARTED","TREATMENT_COMPLETED"),
            ]
 
            for from_s, to_s in transitions:
                if dropped:
                    break
                key = f"{from_s}→{to_s}"
                prob = stage_probs[key]
                d_min, d_max = stage_days[key]
                days_in_stage = random.randint(d_min, d_max)
                current_date = current_date + timedelta(days=days_in_stage)
 
                if random.random() < prob:
                    patient_events.append({
                        "patient_id"      : patient_id,
                        "district_id"     : dist["district_id"],
                        "state"           : dist["state"],
                        "disease"         : disease,
                        "event_type"      : to_s,
                        "event_date"      : str(current_date),
                        "days_from_diag"  : (current_date - diagnosis_date).days,
                        "is_terminal"     : to_s == "TREATMENT_COMPLETED",
                        "is_dropout"      : False,
                        "vulnerability"   : vuln,
                    })
                else:
                    patient_events.append({
                        "patient_id"      : patient_id,
                        "district_id"     : dist["district_id"],
                        "state"           : dist["state"],
                        "disease"         : disease,
                        "event_type"      : "DROPPED_OUT",
                        "event_date"      : str(current_date),
                        "days_from_diag"  : (current_date - diagnosis_date).days,
                        "is_terminal"     : True,
                        "is_dropout"      : True,
                        "vulnerability"   : vuln,
                        "dropout_stage"   : from_s,
                    })
                    dropped = True
            event_id += 1
 
print(f"Patient events generated: {len(patient_events):,}")
 
df_events = spark.createDataFrame(pd.DataFrame(patient_events).fillna(""))
df_events.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.bronze_patient_events")
print("✅ Patient-level event log written")
 


In [0]:
# ── Final summary ────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("BRONZE LAYER COMPLETE")
print("="*55)
for tbl in ["bronze_facility_master","bronze_hmis_referrals",
            "bronze_pmjay_claims","bronze_nfhs_baseline","bronze_patient_events"]:
    cnt = spark.table(f"{DB}.{tbl}").count()
    print(f"  {tbl:<35} {cnt:>8,} rows")
print("="*55)
